# Train and evaluate the handwritten digit demo

Train the CNN, select its checkpoint and acceptance thresholds on validation data, then evaluate separate MNIST and non-digit test sets. The production API loads the saved weights; it never trains on requests. The checked-in checkpoint is already trained. Run the training cell only when you want to retrain.


In [ ]:
from pathlib import Path
import sys, subprocess, json

ROOT = Path.cwd()
if ROOT.name == "training":
    ROOT = ROOT.parent
assert (ROOT / "training/train.py").exists(), "Open this notebook from the repository or training directory."
print(ROOT)


## Install dependencies

The default requirements use CPU PyTorch for portability. For GPU training, install the PyTorch build matching your CUDA environment before running the training script. Training currently runs on CPU; the included run used four threads.


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "training/requirements.txt")])


## Run training

Downloads approximately 11 MB of MNIST and 562 MB of the NIST EMNIST archive into an ignored cache. Training uses 55,000 MNIST digits, 24,000 EMNIST letters, and 12,000 generated invalid examples. Validation and test inputs remain separate. No datasets are committed to Git.


In [ ]:
subprocess.check_call([sys.executable, str(ROOT / "training/train.py"), "--data-dir", str(ROOT / ".cache/datasets"), "--epochs", "8", "--seed", "42", "--threads", "4"], cwd=ROOT)


## Inspect the measured results

Accuracy before rejection is the top digit among classes 0–9. Acceptance also considers the rejection score. Report coverage alongside accepted-digit accuracy. The invalid examples are a defined test distribution, not a proof that every non-digit will be rejected.


In [ ]:
metadata = json.loads((ROOT / "api/artifacts/metadata.json").read_text())
print(json.dumps({"selected_epoch": metadata["selected_epoch"], "thresholds": metadata["thresholds"], "metrics": metadata["metrics"]}, indent=2))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history = metadata["history"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([r["epoch"] for r in history], [r["loss"] for r in history], marker="o")
axes[0].set(xlabel="Epoch", ylabel="Training loss", title="Training history")
matrix = np.array(metadata["metrics"]["confusion_matrix_before_rejection"])
image = axes[1].imshow(matrix, cmap="Blues")
axes[1].set(xlabel="Predicted digit", ylabel="True digit", title="MNIST test confusion matrix", xticks=range(10), yticks=range(10))
fig.colorbar(image, ax=axes[1])
plt.tight_layout()


## Publish the refreshed evaluation summary

Keep the demo’s displayed metrics synchronized when you retrain and redeploy. Check the model version, preprocessing version, weights hash, and API tests before publishing.


In [ ]:
(ROOT / "projects/mnist/evaluation.json").write_text(json.dumps(metadata["metrics"], indent=2) + "\n")
subprocess.check_call([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=ROOT)


## Sources and attribution

- MNIST: https://keras.io/api/datasets/mnist/ — copyright Yann LeCun and Corinna Cortes, CC BY-SA 3.0.
- EMNIST: https://www.nist.gov/itl/products-and-services/emnist-dataset
- Cohen et al. (2017), EMNIST: an extension of MNIST to handwritten letters, https://arxiv.org/abs/1702.05373

Curated example images are selected after evaluation to demonstrate the interface. They are not used for training and do not define the reported benchmark.
